In [23]:
import os
import pandas as pd
import numpy as np
from scipy import stats
from openpyxl import load_workbook
from scipy.spatial.distance import cdist 

# Make sure GOC is bi-directional.

In [ ]:
humanMouseRecipricol = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/humanMouseReciprical.txt", sep="\t").rename(columns={"Gene stable ID": "MouseID", "Human gene stable ID": "HumanID"})

In [ ]:
orthologTable = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.parquet")

In [ ]:
orthologTableHumanGOC = orthologTable.merge(humanMouseRecipricol, left_on=["Gene stable ID", "Mouse gene stable ID"], right_on=["HumanID", "MouseID"], how="left").drop(columns=["MouseID", "HumanID"])
orthologTableHumanGOC.insert(3, "Human Gene-order conservation score", orthologTableHumanGOC.pop("Human Gene-order conservation score"))

# Identifying Duplicated Species

In [ ]:
orthoDist = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.parquet")
display(orthoDist)

In [ ]:
# Identifies the number of mouse genes associated with each human gene.
groupedHumanGene = orthoDist.groupby("Gene stable ID")["Mouse gene stable ID"].apply(list).reset_index(name="Mouse gene stable ID")

# Adds in the homology type to the dataframe.
groupedHumanGene = groupedHumanGene.merge(orthoDist.loc[:, ["Gene stable ID", "Mouse homology type"]])

# Identifies the human genes that have more than one mouse gene associated with them.
groupedHumanGene["Num Mouse Dupes"] = groupedHumanGene["Mouse gene stable ID"].apply(len)

# Creates a new column that identifies which one-to-many gene experienced a duplication event in humans. 
groupedHumanGene["Duplicated Species"] = np.where((groupedHumanGene["Gene stable ID"].isin(groupedHumanGene[groupedHumanGene["Num Mouse Dupes"] > 1]["Gene stable ID"])) & (groupedHumanGene["Mouse homology type"] == "ortholog_one2many"), "Mouse", "NA")

In [ ]:
groupedHumanGene

In [ ]:
# Transfers the information above to the orthologTableDist dataframe.
orthoDistHumanDup = orthoDist.merge(groupedHumanGene.loc[:, ["Gene stable ID", "Duplicated Species", "Num Mouse Dupes"]], left_on="Gene stable ID", right_on="Gene stable ID", how="outer")

In [ ]:
# Same steps as three cells above, but for mouse genes.
groupedMouseGene = orthoDist.groupby("Mouse gene stable ID")["Gene stable ID"].apply(list).reset_index(name="Gene stable ID")
groupedMouseGene = groupedMouseGene.merge(orthoDist.loc[:, ["Mouse gene stable ID", "Mouse homology type"]])
groupedMouseGene["Num Human Dupes"] = groupedMouseGene["Gene stable ID"].apply(len)
groupedMouseGene["Duplicated Species"] = np.where((groupedMouseGene["Mouse gene stable ID"].isin(groupedMouseGene[groupedMouseGene["Num Human Dupes"] > 1]["Mouse gene stable ID"])) & (groupedMouseGene["Mouse homology type"] == "ortholog_one2many"), "Human", "NA")

In [ ]:
groupedMouseGene[groupedMouseGene["Mouse gene stable ID"] == "ENSMUSG00000030945"]

In [ ]:
groupedMouseGene

In [ ]:
# Transfers the information above to the orthologDistTable dataframe.
orthoDistHumanMouseDup = orthoDistHumanDup.merge(groupedMouseGene.loc[:, ["Mouse gene stable ID", "Duplicated Species", "Num Human Dupes"]], on="Mouse gene stable ID", how="outer")

# Because there are two separate "Duplicated Species" columns, I am going to combine their information into one.
# If the human "Duplicated Species" column is "NA" then we use the mouse "Duplicated Species" column. It will either be "Mouse" or stay "NA."
orthoDistHumanMouseDup["Duplicated Species"] = np.where(orthoDistHumanMouseDup["Duplicated Species_x"] == "NA", orthoDistHumanMouseDup["Duplicated Species_y"], orthoDistHumanMouseDup["Duplicated Species_x"])

# Drop the two separate "Duplicated Species" columns.
orthoDistHumanMouseDup = orthoDistHumanMouseDup.loc[:, ~orthoDistHumanMouseDup.columns.isin(["Duplicated Species_x", "Duplicated Species_y"])].drop_duplicates()
display(orthoDistHumanMouseDup)

In [ ]:
orthoDistHumanMouseDup.columns[-1:]

In [ ]:
newColOrder = list(orthoDistHumanMouseDup.columns[:-3]) + list(orthoDistHumanMouseDup.columns[-2:-1]) + list(orthoDistHumanMouseDup.columns[-3:-2]) + list(orthoDistHumanMouseDup.columns[-1:])
orthoDistHumanMouseDup = orthoDistHumanMouseDup.loc[:, newColOrder]

In [ ]:
orthoDistHumanMouseDup.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupe.csv", index=False)
orthoDistHumanMouseDup.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupe.parquet", index=False)

In [ ]:
gtexEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/gtexExpressionProfile.parquet")
emtabEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/emtabExpressionProfile.parquet")

In [ ]:
gtexEP

In [ ]:
emtabEP

In [ ]:
orthoDistHumanMouseDupBioType = orthoDistHumanMouseDup.merge(gtexEP.loc[:, ["Gene type"]], left_on="Gene stable ID", right_index=True, how="left")
orthoDistHumanMouseDupBioType = orthoDistHumanMouseDupBioType.rename(columns={"Gene type": "Human Gene Type"})

In [ ]:
orthoDistHumanMouseDupBioType = orthoDistHumanMouseDupBioType.merge(emtabEP.loc[:, ["Gene type"]], left_on="Mouse gene stable ID", right_index=True, how="left")
orthoDistHumanMouseDupBioType = orthoDistHumanMouseDupBioType.rename(columns={"Gene type": "Mouse Gene Type"})

In [ ]:
orthoDistHumanMouseDupBioType

In [ ]:
orthoDistHumanMouseDupBioType.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.csv")
orthoDistHumanMouseDupBioType.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.parquet")

# Statistical Analysis

In [ ]:
data = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.parquet")
dataProteinOnly = data[(data["Human Gene Type"] == "protein_coding") & (data["Mouse Gene Type"] == "protein_coding")]

In [ ]:
dataProteinOnly

In [ ]:
dataProteinOnly[dataProteinOnly["Mouse homology type"] == "ortholog_one2one"]["EuclidDistNorm"].dropna()

In [ ]:
dataProteinOnly[dataProteinOnly["Mouse homology type"] == "ortholog_one2one"]["TEC"].dropna()

In [ ]:
statsDF = pd.DataFrame({
    "Category": ["one-to-one", 
                 "one-to-many", "one-to-many, duplicated in human", "one-to-many, duplicated in human, GOC=0", "one-to-many, duplicated in human, GOC=25", "one-to-many, duplicated in human, GOC=50", "one-to-many, duplicated in human, GOC=75", "one-to-many, duplicated in human, GOC=100",
                 "one-to-many, duplicated in mouse", "one-to-many, duplicated in mouse, GOC=0", "one-to-many, duplicated in mouse, GOC=25", "one-to-many, duplicated in mouse, GOC=50", "one-to-many, duplicated in mouse, GOC=75", "one-to-many, duplicated in mouse, GOC=100",
                 "many-to-many"],
    "n": "",
    "Median": "",
    "Mean": ""
})

In [ ]:
statsDF

In [ ]:
translationTable = {
    "one-to-one": "ortholog_one2one",
    "one-to-many": "ortholog_one2many",
    "duplicated in human": "Human",
    "duplicated in mouse": "Mouse",
    "GOC=0": 0,
    "GOC=25": 25,
    "GOC=50": 50,
    "GOC=75": 75,
    "GOC=100": 100,
    "many-to-many": "ortholog_many2many"
}

In [ ]:
statsDFArr = []
for distMetric in dataProteinOnly.columns[4:9]:
    nArr = []
    medianArr = []
    meanArr = []

    for category in statsDF["Category"]:
        categoryParts = category.split(", ")
        translatedParts = list(map(translationTable.get, categoryParts))
        if len(translatedParts) == 1:
            filteredDF = dataProteinOnly[dataProteinOnly["Mouse homology type"] == translatedParts[0]][distMetric].dropna()
        elif len(translatedParts) == 2:
            filteredDF = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedParts[0]) & (dataProteinOnly["Duplicated Species"] == translatedParts[1])][distMetric].dropna()
        else:
            filteredDF = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedParts[0]) & (dataProteinOnly["Duplicated Species"] == translatedParts[1]) & (dataProteinOnly["Mouse Gene-order conservation score"] == translatedParts[2])][distMetric].dropna()

        nArr.append(filteredDF.shape[0])
        medianArr.append(filteredDF.median())
        meanArr.append(filteredDF.mean())

    statsDF["n"] = nArr
    statsDF["Median"] = medianArr
    statsDF["Mean"] = meanArr
    statsDFArr.append(statsDF.copy())

In [ ]:
statsDFArr

In [ ]:
pValueDF = pd.DataFrame({
    "Group 1": ["one-to-one", "one-to-one",
                "one-to-many, duplicated in human",
                "many-to-many", "many-to-many",
                "one-to-many, duplicated in human, GOC=0", "one-to-many, duplicated in human, GOC=25", "one-to-many, duplicated in human, GOC=50", "one-to-many, duplicated in human, GOC=75", "one-to-many, duplicated in human, GOC=0",
                "one-to-many, duplicated in mouse, GOC=0", "one-to-many, duplicated in mouse, GOC=25", "one-to-many, duplicated in mouse, GOC=50", "one-to-many, duplicated in mouse, GOC=75", "one-to-many, duplicated in mouse, GOC=0"],

    "Group 2": ["one-to-many, duplicated in human", "one-to-many, duplicated in mouse",
                "one-to-many, duplicated in mouse", 
                "one-to-many, duplicated in human", "one-to-many, duplicated in mouse", 
                "one-to-many, duplicated in human, GOC=25", "one-to-many, duplicated in human, GOC=50", "one-to-many, duplicated in human, GOC=75", "one-to-many, duplicated in human, GOC=100", "one-to-many, duplicated in human, GOC=100",
                "one-to-many, duplicated in mouse, GOC=25", "one-to-many, duplicated in mouse, GOC=50", "one-to-many, duplicated in mouse, GOC=75", "one-to-many, duplicated in mouse, GOC=100", "one-to-many, duplicated in mouse, GOC=100"],
    "P-value": ""
})

In [ ]:
pValueDF

In [ ]:
pValueDFArr = []
for distMetric in dataProteinOnly.columns[4:9]:
    pValueArr = []
    for rowNum in range(pValueDF.shape[0]):
        translatedGroup1 = list(map(translationTable.get, pValueDF.iloc[rowNum, :].iloc[0].split(", ")))
        translatedGroup2 = list(map(translationTable.get, pValueDF.iloc[rowNum, :].iloc[1].split(", ")))

        if len(translatedGroup1) == 1:
            filteredDF1 = dataProteinOnly[dataProteinOnly["Mouse homology type"] == translatedGroup1[0]][distMetric].dropna()
        elif len(translatedGroup1) == 2:
            filteredDF1 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup1[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup1[1])][distMetric].dropna()
        else:
            filteredDF1 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup1[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup1[1]) & (dataProteinOnly["Mouse Gene-order conservation score"] == translatedGroup1[2])][distMetric].dropna()
        
        if len(translatedGroup2) == 1:
            filteredDF2 = dataProteinOnly[dataProteinOnly["Mouse homology type"] == translatedGroup2[0]][distMetric].dropna()
        elif len(translatedGroup2) == 2:
            filteredDF2 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup2[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup2[1])][distMetric].dropna()
        else:
            filteredDF2 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup2[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup2[1]) & (dataProteinOnly["Mouse Gene-order conservation score"] == translatedGroup2[2])][distMetric].dropna()

        pValueArr.append(stats.mannwhitneyu(filteredDF1, filteredDF2)[1])

    pValueDF["P-value"] = pValueArr
    pValueDFArr.append(pValueDF.copy())

In [ ]:
emptyCols = pd.DataFrame({
    "": [np.nan] * len(statsDF),
    " ": [np.nan] * len(statsDF)
})


with pd.ExcelWriter("/Users/andrewhsu/Projects/McNair/data/automatedDistances.xlsx") as w:
    for idx, distMetric in enumerate(dataProteinOnly.columns[4:9]):
        finalDF = pd.concat([statsDFArr[idx].astype(str), emptyCols, pValueDFArr[idx].astype(str)], axis=1)
        finalDF.to_excel(w, sheet_name=distMetric, index=False)

In [ ]:
statsDFArr

# Random Sampling

In [ ]:
# TEC
def TEC(humanOrtholog, mouseOrtholog):
    # Turns the vectors binary. So if the expression is greater than 1, we consider that "expressed."
    humanOrthoBinary = (humanOrtholog.iloc[:, :-1] > 1).iloc[0, :]
    mouseOrthoBinary = (mouseOrtholog.iloc[:, :-1] > 1).iloc[0, :]

    humanOnlyTissueNum = (humanOrthoBinary & ~mouseOrthoBinary).sum()
    mouseOnlyTissueNum = (mouseOrthoBinary & ~humanOrthoBinary).sum()

    humanTotalTissue = humanOrthoBinary.sum()
    mouseTotalTissue = mouseOrthoBinary.sum()

    if humanTotalTissue == 0 and mouseTotalTissue == 0:
        return np.nan
    elif humanTotalTissue == 0 or mouseTotalTissue == 0:
        return np.nan
    else:
        return ((humanOnlyTissueNum / humanTotalTissue) + (mouseOnlyTissueNum / mouseTotalTissue)) / 2


In [ ]:
columns = ["HumanID", "MouseID", "EuclidDist", "EuclidDistNorm", "EuclidDistLog", "PearDist", "TEC"]
sampleDF = pd.DataFrame(columns=columns)

In [ ]:
gtexEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/gtexExpressionProfile.parquet")
emtabEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/emtabExpressionProfile.parquet")

In [ ]:
gtexEPProteinOnly = gtexEP[gtexEP["Gene type"] == "protein_coding"]
emtabEPProteinOnly = emtabEP[emtabEP["Gene type"] == "protein_coding"]

In [ ]:
# humanGenes = gtexEPProteinOnly.sample(10000, replace=False)
# mouseGenes = emtabEPProteinOnly.sample(10000, replace=False)

In [ ]:
# pd.DataFrame({
#     "Human ID": humanGenes.index,
#     "Mouse ID": mouseGenes.index
# }).to_csv("/Users/andrewhsu/Projects/McNair/data/randomGeneIDs.csv", index=False)

In [ ]:
geneIDs = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/randomGeneIDs.csv")

In [ ]:
humanGenes = gtexEPProteinOnly[gtexEPProteinOnly.index.isin(geneIDs["Human ID"])]
mouseGenes = emtabEPProteinOnly[emtabEPProteinOnly.index.isin(geneIDs["Mouse ID"])]

In [ ]:
humanGenes

In [ ]:
humanGenesNorm = humanGenes.iloc[:, :-1].div(np.linalg.norm(humanGenes.iloc[:, :-1], axis=1), axis=0)
mouseGenesNorm = mouseGenes.iloc[:, :-1].div(np.linalg.norm(mouseGenes.iloc[:, :-1], axis=1), axis=0)

In [ ]:
humanGenesLog = np.log2(humanGenes.iloc[:, :-1] + 1)
mouseGenesLog = np.log2(mouseGenes.iloc[:, :-1] + 1)

In [ ]:
euclidDist = np.linalg.norm(humanGenes.iloc[:, :-1] - mouseGenes.iloc[:, :-1].values, axis=1)

In [ ]:
euclidDistNorm = np.linalg.norm(humanGenesNorm - mouseGenesNorm.values, axis=1)

In [ ]:
euclidDistLog = np.linalg.norm(humanGenesLog - mouseGenesLog.values, axis=1)

In [ ]:
humanGenesLog - mouseGenesLog.values

In [ ]:
pearDist = np.diag(cdist(humanGenes.iloc[:, :-1].values.astype(float), mouseGenes.iloc[:, :-1].values.astype(float), metric="correlation"))

In [ ]:
tecValues = [TEC(humanGenes.iloc[rowIdx: rowIdx + 1], mouseGenes.iloc[rowIdx: rowIdx + 1])for rowIdx in range(humanGenes.shape[0])]

In [ ]:
sampleDF["HumanID"] = humanGenes.index
sampleDF["MouseID"] = mouseGenes.index
sampleDF["EuclidDist"] = euclidDist
sampleDF["EuclidDistNorm"] = euclidDistNorm
sampleDF["EuclidDistLog"] = euclidDistLog
sampleDF["PearDist"] = pearDist
sampleDF["TEC"] = tecValues

In [ ]:
sampleDF.to_csv("/Users/andrewhsu/Projects/McNair/data/randomGenesTable.csv", index=False)
sampleDF.to_parquet("/Users/andrewhsu/Projects/McNair/data/randomGenesTable.parquet", index=False)

# Statistical Analysis for Random Samples

In [ ]:
data = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.parquet")
dataProteinOnly = data[(data["Human Gene Type"] == "protein_coding") & (data["Mouse Gene Type"] == "protein_coding")]

randomData = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/randomGenesTable.parquet")

In [ ]:
randomData

In [ ]:
statsDF = pd.DataFrame({
    "Category": ["one-to-one", 
                 "one-to-many", "one-to-many, duplicated in human", "one-to-many, duplicated in human, GOC=0", "one-to-many, duplicated in human, GOC=25", "one-to-many, duplicated in human, GOC=50", "one-to-many, duplicated in human, GOC=75", "one-to-many, duplicated in human, GOC=100",
                 "one-to-many, duplicated in mouse", "one-to-many, duplicated in mouse, GOC=0", "one-to-many, duplicated in mouse, GOC=25", "one-to-many, duplicated in mouse, GOC=50", "one-to-many, duplicated in mouse, GOC=75", "one-to-many, duplicated in mouse, GOC=100",
                 "many-to-many", "random"],
    "n": "",
    "Median": "",
    "Mean": ""
})

pValueDF = pd.DataFrame({
    "Group 1": ["one-to-one", "one-to-one",
                "one-to-many, duplicated in human",
                "many-to-many", "many-to-many",
                "one-to-many, duplicated in human, GOC=0", "one-to-many, duplicated in human, GOC=25", "one-to-many, duplicated in human, GOC=50", "one-to-many, duplicated in human, GOC=75", "one-to-many, duplicated in human, GOC=0",
                "one-to-many, duplicated in mouse, GOC=0", "one-to-many, duplicated in mouse, GOC=25", "one-to-many, duplicated in mouse, GOC=50", "one-to-many, duplicated in mouse, GOC=75", "one-to-many, duplicated in mouse, GOC=0"],

    "Group 2": ["one-to-many, duplicated in human", "one-to-many, duplicated in mouse",
                "one-to-many, duplicated in mouse", 
                "one-to-many, duplicated in human", "one-to-many, duplicated in mouse", 
                "one-to-many, duplicated in human, GOC=25", "one-to-many, duplicated in human, GOC=50", "one-to-many, duplicated in human, GOC=75", "one-to-many, duplicated in human, GOC=100", "one-to-many, duplicated in human, GOC=100",
                "one-to-many, duplicated in mouse, GOC=25", "one-to-many, duplicated in mouse, GOC=50", "one-to-many, duplicated in mouse, GOC=75", "one-to-many, duplicated in mouse, GOC=100", "one-to-many, duplicated in mouse, GOC=100"],
    "P-value": ""
})

In [ ]:
translationTable = {
    "one-to-one": "ortholog_one2one",
    "one-to-many": "ortholog_one2many",
    "duplicated in human": "Human",
    "duplicated in mouse": "Mouse",
    "GOC=0": 0,
    "GOC=25": 25,
    "GOC=50": 50,
    "GOC=75": 75,
    "GOC=100": 100,
    "many-to-many": "ortholog_many2many"
}

In [ ]:
statsDFArr = []
for distMetric in dataProteinOnly.columns[4:9]:
    nArr = []
    medianArr = []
    meanArr = []

    for category in statsDF["Category"]:
        if category == "random":
            filteredDF = randomData[distMetric].dropna()
        else:
            categoryParts = category.split(", ")
            translatedParts = list(map(translationTable.get, categoryParts))
            if len(translatedParts) == 1:
                filteredDF = dataProteinOnly[dataProteinOnly["Mouse homology type"] == translatedParts[0]][distMetric].dropna()
            elif len(translatedParts) == 2:
                filteredDF = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedParts[0]) & (dataProteinOnly["Duplicated Species"] == translatedParts[1])][distMetric].dropna()
            else:
                filteredDF = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedParts[0]) & (dataProteinOnly["Duplicated Species"] == translatedParts[1]) & (dataProteinOnly["Mouse Gene-order conservation score"] == translatedParts[2])][distMetric].dropna()

        nArr.append(filteredDF.shape[0])
        medianArr.append(filteredDF.median())
        meanArr.append(filteredDF.mean())

    statsDF["n"] = nArr
    statsDF["Median"] = medianArr
    statsDF["Mean"] = meanArr
    statsDFArr.append(statsDF.copy())

In [ ]:
pValueDFArr = []
for distMetric in dataProteinOnly.columns[4:9]:
    pValueArr = []
    for rowNum in range(pValueDF.shape[0]):
        translatedGroup1 = list(map(translationTable.get, pValueDF.iloc[rowNum, :].iloc[0].split(", ")))
        translatedGroup2 = list(map(translationTable.get, pValueDF.iloc[rowNum, :].iloc[1].split(", ")))

        if len(translatedGroup1) == 1:
            filteredDF1 = dataProteinOnly[dataProteinOnly["Mouse homology type"] == translatedGroup1[0]][distMetric].dropna()
        elif len(translatedGroup1) == 2:
            filteredDF1 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup1[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup1[1])][distMetric].dropna()
        else:
            filteredDF1 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup1[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup1[1]) & (dataProteinOnly["Mouse Gene-order conservation score"] == translatedGroup1[2])][distMetric].dropna()
        
        if len(translatedGroup2) == 1:
            filteredDF2 = dataProteinOnly[dataProteinOnly["Mouse homology type"] == translatedGroup2[0]][distMetric].dropna()
        elif len(translatedGroup2) == 2:
            filteredDF2 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup2[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup2[1])][distMetric].dropna()
        else:
            filteredDF2 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup2[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup2[1]) & (dataProteinOnly["Mouse Gene-order conservation score"] == translatedGroup2[2])][distMetric].dropna()

        pValueArr.append(stats.mannwhitneyu(filteredDF1, filteredDF2)[1])

    pValueDF["P-value"] = pValueArr
    pValueDFArr.append(pValueDF.copy())

In [ ]:
emptyCols = pd.DataFrame({
    "": [np.nan] * len(statsDF),
    " ": [np.nan] * len(statsDF)
})


with pd.ExcelWriter("/Users/andrewhsu/Projects/McNair/data/automatedDistancesWithRandom.xlsx") as w:
    for idx, distMetric in enumerate(dataProteinOnly.columns[4:9]):
        finalDF = pd.concat([statsDFArr[idx].astype(str), emptyCols, pValueDFArr[idx].astype(str)], axis=1)
        finalDF.to_excel(w, sheet_name=distMetric, index=False)

# Parental-Daughter Identification

In [55]:
humanColumns = ["Human Gene", "Mouse Parent", "Mouse Daughter", "GOC Parent", "GOC Daughter", "EuclidDist Parent", "EuclidDist Daughter", "EuclidDistNorm Parent", "EuclidDistNorm Daughter", "EuclidDistLog Parent", "EuclidDistLog Daughter", "PearDist Parent", "PearDist Daughter", "TEC Parent", "TEC Daughter"]
mouseColumns = ["Mouse Gene", "Human Parent", "Human Daughter", "GOC Parent", "GOC Daughter", "EuclidDist Parent", "EuclidDist Daughter", "EuclidDistNorm Parent", "EuclidDistNorm Daughter", "EuclidDistLog Parent", "EuclidDistLog Daughter", "PearDist Parent", "PearDist Daughter", "TEC Parent", "TEC Daughter"]
humanParent = pd.DataFrame()
mouseParent = pd.DataFrame()

In [56]:
orthologTable = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.parquet")
humanParentTable = orthologTable[(orthologTable["Mouse homology type"] == "ortholog_one2many") & (orthologTable["Duplicated Species"] == "Mouse")]
mouseParentTable = orthologTable[(orthologTable["Mouse homology type"] == "ortholog_one2many") & (orthologTable["Duplicated Species"] == "Human")]

In [57]:
orthologTable["Parental"] = False
orthologTable["Daughter"] = False

In [58]:
humanParentRows = []
for humanID in humanParentTable["Gene stable ID"].unique():
    humanIDDF = humanParentTable[humanParentTable["Gene stable ID"] == humanID]
    humanMaxGOC = humanIDDF["Mouse Gene-order conservation score"].max()
    humanMinGOC = humanIDDF["Mouse Gene-order conservation score"].min()

    if humanMaxGOC != humanMinGOC:
        parentalCopy = humanIDDF[humanIDDF["Mouse Gene-order conservation score"] == humanMaxGOC]
        daughterCopy = humanIDDF[humanIDDF["Mouse Gene-order conservation score"] == humanMinGOC]

        orthologTable.loc[orthologTable["Mouse gene stable ID"].isin(parentalCopy["Mouse gene stable ID"]), "Parental"] = True
        orthologTable.loc[orthologTable["Mouse gene stable ID"].isin(daughterCopy["Mouse gene stable ID"]), "Daughter"] = True

        parentalCopyNAMin = parentalCopy[parentalCopy.isna().sum(axis=1) == parentalCopy.isna().sum(axis=1).min()].sample(n=1)
        daughterCopyNAMin = daughterCopy[daughterCopy.isna().sum(axis=1) == daughterCopy.isna().sum(axis=1).min()].sample(n=1)

        humanParentRow = [humanID, parentalCopyNAMin["Mouse gene stable ID"].item(), daughterCopyNAMin["Mouse gene stable ID"].item(), 
                        parentalCopyNAMin["Mouse Gene-order conservation score"].item(), daughterCopyNAMin["Mouse Gene-order conservation score"].item(),
                        parentalCopyNAMin["EuclidDist"].item(), daughterCopyNAMin["EuclidDist"].item(),
                        parentalCopyNAMin["EuclidDistNorm"].item(), daughterCopyNAMin["EuclidDistNorm"].item(),
                        parentalCopyNAMin["EuclidDistLog"].item(), daughterCopyNAMin["EuclidDistLog"].item(),
                        parentalCopyNAMin["PearDist"].item(), daughterCopyNAMin["PearDist"].item(),
                        parentalCopyNAMin["TEC"].item(), daughterCopyNAMin["TEC"].item()]
        humanParentRows.append(humanParentRow)
humanParent = pd.DataFrame(humanParentRows)
humanParent.columns = humanColumns

In [59]:
humanParent

,Human Gene,Mouse Parent,Mouse Daughter,GOC Parent,GOC Daughter,EuclidDist Parent,EuclidDist Daughter,EuclidDistNorm Parent,EuclidDistNorm Daughter,EuclidDistLog Parent,EuclidDistLog Daughter,PearDist Parent,PearDist Daughter,TEC Parent,TEC Daughter
0,ENSG00000254647,ENSMUSG00000000215,ENSMUSG00000035804,100.0,0.0,3667.846303,5264.286219,0.001498,0.001428,2.445157,4.066262,5.970543e-07,8.228542e-07,0.583333,0.250000
1,ENSG00000069493,ENSMUSG00000030157,ENSMUSG00000030365,50.0,0.0,495.295522,27.050040,0.940933,1.128786,11.994275,4.207646,1.027792e+00,9.907103e-01,0.000000,0.375000
2,ENSG00000157601,ENSMUSG00000023341,ENSMUSG00000000386,75.0,25.0,31.889421,31.041610,0.814844,0.718185,5.317110,6.218484,1.319186e+00,7.603217e-01,0.062500,0.187500
3,ENSG00000146425,ENSMUSG00000095677,ENSMUSG00000096255,50.0,0.0,346.575685,344.134492,0.499185,0.488780,13.120709,11.645987,4.088576e-01,6.034510e-01,0.187500,0.000000
4,ENSG00000089127,ENSMUSG00000066861,ENSMUSG00000001168,100.0,0.0,27.405453,20.208340,0.945163,0.943921,4.503699,6.439363,6.577287e-01,7.247547e-01,0.142857,0.428571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,ENSG00000151327,ENSMUSG00000094103,ENSMUSG00000095595,75.0,50.0,34.377397,34.377397,NaN,NaN,9.680407,9.680407,NaN,NaN,NaN,NaN
223,ENSG00000026297,ENSMUSG00000095687,ENSMUSG00000094724,50.0,25.0,11.552547,12.033663,0.652015,0.666927,2.772407,2.871943,8.719934e-01,9.140217e-01,0.062500,0.062500
224,ENSG00000188324,ENSMUSG00000095401,ENSMUSG00000095075,25.0,0.0,0.009028,0.009028,NaN,NaN,0.012357,0.012357,NaN,NaN,NaN,NaN
225,ENSG00000146385,ENSMUSG00000100004,ENSMUSG00000096442,100.0,0.0,0.070029,0.021049,1.314738,NaN,0.097703,0.029916,1.205051e+00,NaN,NaN,NaN


In [60]:
humanParent.to_csv("/Users/andrewhsu/Projects/McNair/data/mouseParentDaughter.csv", index=False)
humanParent.to_parquet("/Users/andrewhsu/Projects/McNair/data/mouseParentDaughter.parquet", index=False)

In [61]:
humanParent

,Human Gene,Mouse Parent,Mouse Daughter,GOC Parent,GOC Daughter,EuclidDist Parent,EuclidDist Daughter,EuclidDistNorm Parent,EuclidDistNorm Daughter,EuclidDistLog Parent,EuclidDistLog Daughter,PearDist Parent,PearDist Daughter,TEC Parent,TEC Daughter
0,ENSG00000254647,ENSMUSG00000000215,ENSMUSG00000035804,100.0,0.0,3667.846303,5264.286219,0.001498,0.001428,2.445157,4.066262,5.970543e-07,8.228542e-07,0.583333,0.250000
1,ENSG00000069493,ENSMUSG00000030157,ENSMUSG00000030365,50.0,0.0,495.295522,27.050040,0.940933,1.128786,11.994275,4.207646,1.027792e+00,9.907103e-01,0.000000,0.375000
2,ENSG00000157601,ENSMUSG00000023341,ENSMUSG00000000386,75.0,25.0,31.889421,31.041610,0.814844,0.718185,5.317110,6.218484,1.319186e+00,7.603217e-01,0.062500,0.187500
3,ENSG00000146425,ENSMUSG00000095677,ENSMUSG00000096255,50.0,0.0,346.575685,344.134492,0.499185,0.488780,13.120709,11.645987,4.088576e-01,6.034510e-01,0.187500,0.000000
4,ENSG00000089127,ENSMUSG00000066861,ENSMUSG00000001168,100.0,0.0,27.405453,20.208340,0.945163,0.943921,4.503699,6.439363,6.577287e-01,7.247547e-01,0.142857,0.428571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,ENSG00000151327,ENSMUSG00000094103,ENSMUSG00000095595,75.0,50.0,34.377397,34.377397,NaN,NaN,9.680407,9.680407,NaN,NaN,NaN,NaN
223,ENSG00000026297,ENSMUSG00000095687,ENSMUSG00000094724,50.0,25.0,11.552547,12.033663,0.652015,0.666927,2.772407,2.871943,8.719934e-01,9.140217e-01,0.062500,0.062500
224,ENSG00000188324,ENSMUSG00000095401,ENSMUSG00000095075,25.0,0.0,0.009028,0.009028,NaN,NaN,0.012357,0.012357,NaN,NaN,NaN,NaN
225,ENSG00000146385,ENSMUSG00000100004,ENSMUSG00000096442,100.0,0.0,0.070029,0.021049,1.314738,NaN,0.097703,0.029916,1.205051e+00,NaN,NaN,NaN


In [62]:
mouseParentRows = []
for mouseID in mouseParentTable["Mouse gene stable ID"].unique():
    mouseIDDF = mouseParentTable[mouseParentTable["Mouse gene stable ID"] == mouseID]
    mouseMaxGOC = mouseIDDF["Mouse Gene-order conservation score"].max()
    mouseMinGOC = mouseIDDF["Mouse Gene-order conservation score"].min()

    if mouseMaxGOC != mouseMinGOC:
        parentalCopy = mouseIDDF[mouseIDDF["Mouse Gene-order conservation score"] == mouseMaxGOC]
        daughterCopy = mouseIDDF[mouseIDDF["Mouse Gene-order conservation score"] == mouseMinGOC]

        orthologTable.loc[orthologTable["Gene stable ID"].isin(parentalCopy["Gene stable ID"]), "Parental"] = True
        orthologTable.loc[orthologTable["Gene stable ID"].isin(daughterCopy["Gene stable ID"]), "Daughter"] = True

        parentalCopyNAMin = parentalCopy[parentalCopy.isna().sum(axis=1) == parentalCopy.isna().sum(axis=1).min()].sample(n=1)
        daughterCopyNAMin = daughterCopy[daughterCopy.isna().sum(axis=1) == daughterCopy.isna().sum(axis=1).min()].sample(n=1)

        mouseParentRow = [mouseID, parentalCopyNAMin["Gene stable ID"].item(), daughterCopyNAMin["Gene stable ID"].item(), 
                        parentalCopyNAMin["Mouse Gene-order conservation score"].item(), daughterCopyNAMin["Mouse Gene-order conservation score"].item(),
                        parentalCopyNAMin["EuclidDist"].item(), daughterCopyNAMin["EuclidDist"].item(),
                        parentalCopyNAMin["EuclidDistNorm"].item(), daughterCopyNAMin["EuclidDistNorm"].item(),
                        parentalCopyNAMin["EuclidDistLog"].item(), daughterCopyNAMin["EuclidDistLog"].item(),
                        parentalCopyNAMin["PearDist"].item(), daughterCopyNAMin["PearDist"].item(),
                        parentalCopyNAMin["TEC"].item(), daughterCopyNAMin["TEC"].item()]
        mouseParentRows.append(mouseParentRow)
mouseParent = pd.DataFrame(mouseParentRows)
mouseParent.columns = mouseColumns

In [63]:
mouseParent.to_csv("/Users/andrewhsu/Projects/McNair/data/humanParentDaughter.csv", index=False)
mouseParent.to_parquet("/Users/andrewhsu/Projects/McNair/data/humanParentDaughter.parquet", index=False)

In [64]:
mouseParent

,Mouse Gene,Human Parent,Human Daughter,GOC Parent,GOC Daughter,EuclidDist Parent,EuclidDist Daughter,EuclidDistNorm Parent,EuclidDistNorm Daughter,EuclidDistLog Parent,EuclidDistLog Daughter,PearDist Parent,PearDist Daughter,TEC Parent,TEC Daughter
0,ENSMUSG00000000308,ENSG00000237289,ENSG00000223572,100.0,75.0,2561.844409,2569.930198,1.182314,0.869675,11.026921,12.033288,0.921149,0.554442,0.187500,0.187500
1,ENSMUSG00000000804,ENSG00000170832,ENSG00000129204,50.0,0.0,94.660307,115.484260,0.495699,0.769929,4.109197,10.174987,0.335662,0.869201,0.000000,0.375000
2,ENSMUSG00000000982,ENSG00000277632,ENSG00000275385,25.0,0.0,9.978136,10.376588,0.789561,1.049001,4.253360,4.076298,1.002236,1.147996,0.428571,0.333333
3,ENSMUSG00000001157,ENSG00000087338,ENSG00000244234,100.0,0.0,43.865728,60.650177,0.472288,0.727488,4.197032,11.185014,0.673056,1.021586,0.000000,NaN
4,ENSMUSG00000002546,ENSG00000167110,ENSG00000232653,75.0,0.0,60.192802,129.935081,0.436744,0.736270,3.662404,9.381758,0.358321,1.210480,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,ENSMUSG00000102976,ENSG00000058673,ENSG00000214558,100.0,0.0,292.704208,292.634675,0.555819,0.478797,13.873760,13.724365,0.678586,0.470963,0.312500,0.312500
123,ENSMUSG00000103707,ENSG00000081842,ENSG00000204962,75.0,50.0,2.791802,3.229265,0.456832,0.289487,1.416028,1.722690,0.061593,0.022382,NaN,NaN
124,ENSMUSG00000116652,ENSG00000286025,ENSG00000286102,25.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125,ENSMUSG00000121607,ENSG00000146938,ENSG00000165246,25.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [65]:
humanStats = {}
for distMetric in humanParent.columns[5::2]:
    numParentalHigher = (humanParent[distMetric] > humanParent[distMetric.replace("Parent", "Daughter")]).sum()
    numDaughterHigher = (humanParent[distMetric] < humanParent[distMetric.replace("Parent", "Daughter")]).sum()
    numEqual = (humanParent[distMetric] == humanParent[distMetric.replace("Parent", "Daughter")]).sum()
    humanStats[distMetric.replace(" Parent", "")] = [numParentalHigher, numDaughterHigher, numEqual]

humanStatsDF = pd.DataFrame(humanStats)
humanStatsDF.index = ["Parental", "Daughter", "Equal"]

In [66]:
humanStatsDF

,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC
Parental,76,38,49,51,12
Daughter,95,97,122,84,42
Equal,39,4,39,4,11


In [67]:
mouseStats = {}
for distMetric in mouseParent.columns[5::2]:
    numParentalHigher = (mouseParent[distMetric] > mouseParent[distMetric.replace("Parent", "Daughter")]).sum()
    numDaughterHigher = (mouseParent[distMetric] < mouseParent[distMetric.replace("Parent", "Daughter")]).sum()
    numEqual = (mouseParent[distMetric] == mouseParent[distMetric.replace("Parent", "Daughter")]).sum()
    mouseStats[distMetric.replace(" Parent", "")] = [numParentalHigher, numDaughterHigher, numEqual]

mouseStatsDF = pd.DataFrame(mouseStats)
mouseStatsDF.index = ["Parental", "Daughter", "Equal"]

In [68]:
mouseStatsDF

,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC
Parental,48,45,45,44,4
Daughter,72,62,75,63,18
Equal,0,0,0,0,19


In [69]:
humanStatsT = humanStatsDF.T.iloc[:, :-1]
humanStatsT["pValue"] = [stats.binomtest(k, n, alternative="greater").pvalue for k, n in zip(humanStatsT["Daughter"], humanStatsT["Daughter"] + humanStatsT["Parental"])]

mouseStatsT = mouseStatsDF.T.iloc[:, :-1]
mouseStatsT["pValue"] = [stats.binomtest(k, n, alternative="greater").pvalue for k, n in zip(mouseStatsT["Daughter"], mouseStatsT["Daughter"] + mouseStatsT["Parental"])]

In [70]:
with pd.ExcelWriter("/Users/andrewhsu/Projects/McNair/data/parentDaughterStats.xlsx") as w:
    humanStatsT.T.astype(str).to_excel(w, sheet_name="Mouse", index=False)
    mouseStatsT.T.astype(str).to_excel(w, sheet_name="Human", index=False)

In [76]:
orthologTable.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioTypeParentalDaughter.csv")
orthologTable.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioTypeParentalDaughter.parquet")

# Parental-Daughter Analysis

In [2]:
orthoTable = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioTypeParentalDaughter.parquet")

In [3]:
orthoTable

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC,Num Human Dupes,Num Mouse Dupes,Duplicated Species,Human Gene Type,Mouse Gene Type,Parental,Daughter
0,ENSG00000065135,ENSMUSG00000000001,ortholog_one2one,100.0,197.746445,0.278561,6.701999,0.151442,0.000000,1.0,1,NA,protein_coding,protein_coding,False,False
1,ENSG00000093009,ENSMUSG00000000028,ortholog_one2one,100.0,24.533616,1.024816,5.058507,1.001638,0.250000,1.0,1,NA,protein_coding,protein_coding,False,False
2,ENSG00000102098,ENSMUSG00000000037,ortholog_one2one,100.0,2.084639,0.471628,1.444542,0.161786,0.166667,1.0,1,NA,protein_coding,protein_coding,False,False
3,ENSG00000091583,ENSMUSG00000000049,ortholog_one2one,100.0,1776.546401,0.002316,2.516381,0.000003,0.500000,1.0,1,NA,protein_coding,protein_coding,False,False
4,ENSG00000141562,ENSMUSG00000000056,ortholog_one2one,100.0,70.950296,0.525139,3.511421,0.661562,0.000000,1.0,1,NA,protein_coding,protein_coding,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3884923,ENSG00000310562,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,False,False
3884924,ENSG00000310579,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,False,False
3884925,ENSG00000310583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,False,False
3884926,ENSG00000310590,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,False,False


In [4]:
parentalDaughterDF = pd.DataFrame({
    "Categories": ["trios, duplicated in human, parental", "trios, duplicated in human, daughter",
                   "trios, duplicated in mouse, parental", "trios, duplicated in mouse, daughter"],
    "n": "",
    "Median": "",
    "Mean": ""
})

In [11]:
translationTable = {
    "trios": "ortholog_one2many",
    "duplicated in human": "Human",
    "duplicated in mouse": "Mouse",
    "parental": "Parental",
    "daughter": "Daughter"
}

In [15]:
masterDF = []
for distMetric in orthoTable.columns[4:9]:
    nArr = []
    medianArr = []
    meanArr = []
    for category in parentalDaughterDF["Categories"]:
        translatedCats = list(map(translationTable.get, category.split(", ")))
        n = orthoTable[(orthoTable["Mouse homology type"] == translatedCats[0]) & (orthoTable["Duplicated Species"] == translatedCats[1]) & (orthoTable[translatedCats[2]] == True)][distMetric].dropna().shape[0]
        median = orthoTable[(orthoTable["Mouse homology type"] == translatedCats[0]) & (orthoTable["Duplicated Species"] == translatedCats[1]) & (orthoTable[translatedCats[2]] == True)][distMetric].dropna().median()
        mean = orthoTable[(orthoTable["Mouse homology type"] == translatedCats[0]) & (orthoTable["Duplicated Species"] == translatedCats[1]) & (orthoTable[translatedCats[2]] == True)][distMetric].dropna().mean()

        nArr.append(n)
        medianArr.append(median)
        meanArr.append(mean)

    parentalDaughterDF["n"] = nArr
    parentalDaughterDF["Median"] = medianArr
    parentalDaughterDF["Mean"] = meanArr
    masterDF.append(parentalDaughterDF.copy())

In [25]:
with pd.ExcelWriter("/Users/andrewhsu/Projects/McNair/data/automatedDistancesWithRandom copy.xlsx", mode="a", if_sheet_exists="overlay") as w:
    for idx, distMetric in enumerate(orthoTable.columns[4:9]):
        masterDF[idx].to_excel(w, sheet_name=distMetric, startrow=load_workbook("/Users/andrewhsu/Projects/McNair/data/automatedDistancesWithRandom copy.xlsx")[distMetric].max_row, header=False, index=False)

os.rename("/Users/andrewhsu/Projects/McNair/data/automatedDistancesWithRandom copy.xlsx", "/Users/andrewhsu/Projects/McNair/data/automatedDistancesWithRandomAndParentalDaughter.xlsx")

# GO Enrichment Analysis

In [72]:
oneToManyGO = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/oneToManyOrtho.txt", sep="\t", skiprows=11)
oneToOneGO = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/oneToOneOrtho.txt", sep="\t", skiprows=11)

In [73]:
display(oneToManyGO)
display(oneToOneGO)

,GO biological process complete,Homo sapiens - REFLIST (20580),upload_1 (794),upload_1 (expected),upload_1 (over/under),upload_1 (fold Enrichment),upload_1 (raw P-value),upload_1 (FDR)
0,absorption of visible light (GO:0016038),6,4,0.23,+,17.28,3.100000e-05,0.003520
1,light absorption (GO:0016037),6,4,0.23,+,17.28,3.100000e-05,0.003490
2,adenosine 5'-(hexahydrogen pentaphosphate) cat...,5,3,0.19,+,15.55,5.400000e-04,0.037500
3,diadenosine hexaphosphate catabolic process (G...,5,3,0.19,+,15.55,5.400000e-04,0.037400
4,diadenosine pentaphosphate catabolic process (...,5,3,0.19,+,15.55,5.400000e-04,0.037200
...,...,...,...,...,...,...,...,...
226,inner ear development (GO:0048839),205,0,7.91,-,< 0.01,6.900000e-04,0.044400
227,positive regulation of neurogenesis (GO:0050769),217,0,8.37,-,< 0.01,2.940000e-04,0.023900
228,ear development (GO:0043583),235,0,9.07,-,< 0.01,2.120000e-04,0.017800
229,camera-type eye development (GO:0043010),368,0,14.20,-,< 0.01,7.830000e-07,0.000158


,GO biological process complete,Homo sapiens - REFLIST (20580),upload_1 (5113),upload_1 (expected),upload_1 (over/under),upload_1 (fold Enrichment),upload_1 (raw P-value),upload_1 (FDR)
0,activation-induced cell death of T cells (GO:0...,5,5,1.24,+,4.03,0.000945,0.02400
1,CD27 signaling pathway (GO:0160162),5,5,1.24,+,4.03,0.000945,0.02400
2,protein ufmylation (GO:0071569),7,7,1.74,+,4.03,0.000058,0.00233
3,positive regulation of monocyte extravasation ...,5,5,1.24,+,4.03,0.000945,0.02390
4,protein K69-linked ufmylation (GO:1990592),5,5,1.24,+,4.03,0.000945,0.02390
...,...,...,...,...,...,...,...,...
668,telencephalon glial cell migration (GO:0022030),28,0,6.96,-,< 0.01,0.000600,0.01650
669,positive regulation of neurotransmitter transp...,26,0,6.46,-,< 0.01,0.000952,0.02390
670,regulation of long-term neuronal synaptic plas...,26,0,6.46,-,< 0.01,0.000952,0.02390
671,synaptic vesicle maturation (GO:0016188),27,0,6.71,-,< 0.01,0.000593,0.01650


In [74]:
oneToManyGO.merge(oneToOneGO, on="GO biological process complete", how="inner", suffixes=("_one2many", "_one2one")).sort_values(by=["upload_1 (fold Enrichment)_one2many"])

,GO biological process complete,Homo sapiens - REFLIST (20580)_one2many,upload_1 (794),upload_1 (expected)_one2many,upload_1 (over/under)_one2many,upload_1 (fold Enrichment)_one2many,upload_1 (raw P-value)_one2many,upload_1 (FDR)_one2many,Homo sapiens - REFLIST (20580)_one2one,upload_1 (5113),upload_1 (expected)_one2one,upload_1 (over/under)_one2one,upload_1 (fold Enrichment)_one2one,upload_1 (raw P-value)_one2one,upload_1 (FDR)_one2one
77,axonogenesis (GO:0007409),368,2,14.20,-,.14,1.010000e-04,9.400000e-03,368,55,91.43,-,.60,4.400000e-06,2.290000e-04
76,cell morphogenesis involved in neuron differen...,448,3,17.28,-,.17,5.130000e-05,5.560000e-03,448,72,111.30,-,.65,6.950000e-06,3.480000e-04
74,axon development (GO:0061564),427,3,16.47,-,.18,1.030000e-04,9.500000e-03,427,63,106.09,-,.59,4.040000e-07,2.600000e-05
75,regulation of neuron projection development (G...,435,3,16.78,-,.18,7.080000e-05,7.000000e-03,435,71,108.07,-,.66,1.880000e-05,8.550000e-04
71,cell morphogenesis (GO:0000902),715,7,27.59,-,.25,3.920000e-06,6.550000e-04,715,132,177.64,-,.74,4.050000e-05,1.690000e-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,sensory perception of chemical stimulus (GO:00...,548,108,21.14,+,5.11,2.360000e-46,6.870000e-43,548,51,136.15,-,.37,7.700000e-21,3.730000e-18
3,detection of chemical stimulus (GO:0009593),522,107,20.14,+,5.31,1.210000e-47,4.410000e-44,522,42,129.69,-,.32,1.670000e-23,9.730000e-21
2,detection of chemical stimulus involved in sen...,485,107,18.71,+,5.72,6.320000e-51,4.600000e-47,485,33,120.50,-,.27,1.380000e-25,9.090000e-23
1,sensory perception of smell (GO:0007608),467,104,18.02,+,5.77,6.820000e-50,3.310000e-46,467,30,116.02,-,.26,5.470000e-26,3.780000e-23
